### Setup
Autoreload and imports

In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Imports
import sys, os
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method

sys.path.append(str(Path.cwd().parents[2]))

from utils.functions import get_windowed_data
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader

### Load Model and Data
Load the data and model and run on clean data to check the accuracy/F1.

In [ ]:
## Inputs
checkpoint_name, data_name = FilenameLoader.rand_pos()

checkpoint_file= f"../../../saved_models/{checkpoint_name}"
data_file = f"../../../data/{data_name}"
save_path = "../final_data/randpos"

In [ ]:
## Load model and data
# Load original model and ART-wrapper classifier
model, classifier = get_model_classifier(checkpoint_file)

# Load data
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)

In [ ]:
## Run clean data test
out = clean_data_test(model = model, classifier = classifier, # model information
                x_test=x_test, y_test=y_test, # data information
                checkpoint_file=checkpoint_file, data_file=data_file, # to save in json
                save_path=save_path, filename="clean.json", # saving information
                save_results=True
                ) 

# Check if output passes
print('noWrapper:', 'PASS' if out['noWrapper']['f1'] > 0.85 else 'FAIL')
print('wrapper:', 'PASS' if out['wrapper']['f1'] > 0.85 else 'FAIL')

In [ ]:


# for i in range(len(vehicles)):
#     vehicle_file = vehicles[i]
#     model, classifier = get_classifier(vehicle_file)
#     benign_test(in_model=model,
#                 in_classifier = classifier,
#                 save_path="./data-test",
#                 filename = f"benign_{prettify_filename(vehicle_file)}.json", 
#                 checkpoint_file=vehicle_file)
    
for i in range(len(vehicles)):
    vehicle_file = vehicles[i]
    model, classifier = get_classifier(vehicle_file)

    save_path = "../data-test/randpos/fgsm"
    filename = f"adv_eps_0.1_{prettify_filename(vehicle_file)}.json"
    adv_test(end_index = len(y_test.numpy()),
        path = save_path,
        filename=filename,
        in_classifier=classifier,
        Attack=FastGradientMethod,
        eps=0.1,
        )

### Adversarial training

For each vehicle checkpoint, fine-tunes the model on a mix of clean and FGSM-adversarial
examples (`AdversarialTrainer`), then re-evaluates benign and adversarial performance and
saves the adversarially-trained weights alongside the metrics.

In [ ]:
# Adversarial training: fine-tune each vehicle's model on clean + FGSM examples, then re-evaluate
from art.defences.trainer import AdversarialTrainer

adv_train_eps = 0.1
adv_train_epochs = 5
adv_train_ratio = 0.5  # fraction of each batch replaced with adversarial examples
adv_save_dir = "../data-test/randpos/adv_final_ckpt"
os.makedirs(adv_save_dir, exist_ok=True)

x_train_np = x_train.numpy()
y_train_np = y_train.numpy()

checkpoint_file="../../saved_models/RandomPos-final.ckpt"
name = prettify_filename(checkpoint_file)

print(f"=== Adversarial training: {name} ===")

benign_test(
    in_model=model,
    in_classifier=classifier,
    save_path=adv_save_dir,
    filename=f"benign_before_advtrained_{name}.json",
    checkpoint_file=vehicle_file,
)
adv_test(
    end_index=len(y_test.numpy()),
    path=adv_save_dir,
    filename=f"adv_before_eps_{adv_train_eps}_advtrained_{name}.json",
    in_classifier=classifier,
    Attack=FastGradientMethod,
    eps=adv_train_eps,
)

attack = FastGradientMethod(classifier, eps=adv_train_eps)
trainer = AdversarialTrainer(classifier, attacks=attack, ratio=adv_train_ratio)
trainer.fit(x_train_np, y_train_np, nb_epochs=adv_train_epochs)

# Re-evaluate after adversarial training
benign_test(
    in_model=model,
    in_classifier=classifier,
    save_path=adv_save_dir,
    filename=f"benign_after_advtrained_{name}.json",
    checkpoint_file=vehicle_file,
)
adv_test(
    end_index=len(y_test.numpy()),
    path=adv_save_dir,
    filename=f"adv_after_eps_{adv_train_eps}_advtrained_{name}.json",
    in_classifier=classifier,
    Attack=FastGradientMethod,
    eps=adv_train_eps,
)

# Save the adversarially-trained weights (loadable via utils.functions.load_model_checkpoint)
ckpt_out = f"{adv_save_dir}/advtrained_{name}.ckpt"
torch.save(model.learner.state_dict(), ckpt_out)
print(f"Saved adversarially-trained checkpoint to {ckpt_out}")

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!
=== Adversarial training: RandomPos-final ===
Saved to ../data-test/randpos/adv_final_ckpt/benign_before_advtrained_RandomPos-final.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.1} ===
Accuracy:           0.3487
Precision:          0.2855
Recall:             0.7934
F1:                 0.4199
ASR (FNR):          0.2066
False Positive Rate:0.8393
TP=294115, TN=140982, FP=736058, FN=76585
Time elapsed:       17.24s
Saved metrics to ../data-test/randpos/adv_final_ckpt/adv_before_eps_0.1_advtrained_RandomPos-final.json


Adversarial training epochs: 100%|██████████| 5/5 [04:01<00:00, 48.21s/it]


Saved to ../data-test/randpos/adv_final_ckpt/benign_after_advtrained_RandomPos-final.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.1} ===
Accuracy:           0.9903
Precision:          0.9925
Recall:             0.9747
F1:                 0.9835
ASR (FNR):          0.0253
False Positive Rate:0.0031
TP=361305, TN=874315, FP=2725, FN=9395
Time elapsed:       17.11s
Saved metrics to ../data-test/randpos/adv_final_ckpt/adv_after_eps_0.1_advtrained_RandomPos-final.json
Saved adversarially-trained checkpoint to ../data-test/randpos/adv_final_ckpt/advtrained_RandomPos-final.ckpt
